# Danish vs TARTS Wavefront Retrieval Comparison — 20260713 (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-08-10
**Last Modified:** 2026-08-10
**Status:** In Progress
**Keywords:** AOS, corner WFS, TARTS, Danish, Zernikes, v-modes, block-T668

## Description

Compare three corner-wavefront-sensor (CWFS) Zernike-retrieval methods on the
20260713 TARTS test (block-T668):

1. **TARTS** — `u/peterma2/tarts_july14` (ML-combined `aggregateAOSVisitTableAvg`, one row per detector).
2. **Danish paired** — `.../cwfs/danish_1_2_0/wep_17_7_0/dv_4_7_0/bin_x2/paired/block-T668/refitWcs`.
3. **Danish unpaired** — `.../cwfs/danish_1_2_0/.../unpaired/block-T668/refitWcs` (each side of focus + mean).

Key functionality:
1. Zernike time-histories (Z4–Z26 excl. Z20,Z21) vs `seq_num`, one panel per Zj, all methods overlaid.
2. v-modes for each method in both (22 DOF, 12 v-modes) and (50 DOF, 34 v-modes) schemes, vs `seq_num`.
3. Scatter TARTS vs Danish-paired for each Zj (21 panels).
4. Scatter TARTS vs Danish-paired v-modes (12 and 34).
5. Time-history of the 22-DOF Trim (aggregatedDoF) and delivered FWHM (ConsDB) vs `seq_num`.

**Output:** inline plots + PDFs under `output/`.

**Runs on:** USDF RSP (needs the LSST stack, `/repo/embargo` or `/repo/main`, EFD + ConsDB).

**Reuses:** `aos/code/aos_state.py` (`build_geom_svd`, `recover_optical_state`,
`DOF_SETS`, `ZK_NOLL`, `SENSOR_NAMES`), `aos/code/aos_trim.py`
(`fetch_aggregated_dof_for_visits`, `make_consdb_client`),
`aos/code/run_wfs_mktable.py` (`get_unpaired_zernikes`),
`aos/code/run_wfs_dof_compare.py` (`robust_fit`, `nmad`), and
`lsst.ts.intrinsic.wavefront.{intrinsics_lib.get_aggregate_zernikes, ofc_svd.build_ofc_svd}`.

**Based on:** `run_wfs_dof_compare.py` (v-mode schemes), `blocks/code/build_night_table.py` (ConsDB FWHM).


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-08-10 | Aaron Roodman | Initial version — TARTS vs Danish paired/unpaired comparison |
| 2026-08-10 | Aaron Roodman | Unpaired "mean" = straight side-mean; add per-corner plots + TARTS intrinsic overlay; v-modes use a common (TARTS) intrinsic deviation |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access — load per-method corner Zernikes](#data)
5. [1. Zernike time histories](#zk-timehist)
6. [2. v-modes: compute + time histories](#vmode-timehist)
7. [3. Zernike scatter: TARTS vs Danish paired](#zk-scatter)
8. [4. v-mode scatter: TARTS vs Danish paired](#vmode-scatter)
9. [5. Trim (22 DOF) & delivered FWHM](#trim-fwhm)


<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

DAY_OBS = 20260713                      # TARTS test night (block-T668)
COORD   = "OCS"                         # coordinate system for Zernikes (OCS or CCS)

# Butler repo. The Danish CWFS collections were written to /repo/embargo; if they
# have since been transferred, switch to "/repo/main".
BUTLER_REPO = "/repo/embargo"

# The three wavefront-retrieval collections + how to read each one.
#   reader:       'paired'  -> intrinsics_lib.get_aggregate_zernikes (intra/extra pair)
#                 'unpaired'-> run_wfs_mktable.get_unpaired_zernikes (singular position)
#   dataset_type: Butler dataset type to read (Avg = TARTS ML-combined, one row/detector)
METHODS = {
    "TARTS": dict(
        collection="u/peterma2/tarts_july14",
        reader="unpaired",
        dataset_type="aggregateAOSVisitTableAvg",
        color="crimson",
    ),
    "Danish paired": dict(
        collection="LSSTCam/runs/aos/cwfs/danish_1_2_0/wep_17_7_0/dv_4_7_0/"
                   "bin_x2/paired/block-T668/refitWcs",
        reader="paired",
        dataset_type="aggregateAOSVisitTableRaw",
        color="steelblue",
    ),
    "Danish unpaired": dict(
        collection="LSSTCam/runs/aos/cwfs/danish_1_2_0/wep_17_7_0/dv_4_7_0/"
                   "bin_x2/unpaired/block-T668/refitWcs",
        reader="unpaired",
        dataset_type="aggregateAOSVisitTableRaw",
        color="seagreen",
    ),
}

# For "Danish unpaired", split the two sides of focus by the corner half-sensor.
# Each corner raft has SW0 and SW1 chips at opposite piston offsets; one is the
# extra-focal side, the other intra-focal. (Flip if the sign convention is reversed.)
UNPAIRED_EXTRA_SUFFIX = "SW0"           # side-of-focus A
UNPAIRED_INTRA_SUFFIX = "SW1"           # side-of-focus B

# How to combine the two focus sides of Danish unpaired into its "mean":
#   "sidemean" -> straight mean of the intra-side and extra-side per-corner medians
#   "pooled"   -> median over all donuts of both sides (donut-count weighted; the
#                 original behavior — not a straight mean)
UNPAIRED_MEAN_MODE = "sidemean"

# Intrinsic wavefront used to form the per-corner DEVIATION for the v-modes:
#   "TARTS" -> use the intrinsic tabulated in the TARTS aggregate table for ALL
#              methods (a single common intrinsic; recommended for this study)
#   "self"  -> each method subtracts its own tabulated intrinsic
INTRINSIC_SOURCE = "TARTS"

PLOT_PER_CORNER = True                  # also make per-corner (not just corner-mean) plots

# v-mode schemes: (dof_set in aos_state.DOF_SETS, n_vmodes)
VMODE_SCHEMES = {
    "22DOF-12vmode": ("standard_22", 12),
    "50DOF-34vmode": ("all_50",      34),
}

SCATTER_REJECT_K = 5.0                  # nMAD outlier factor for robust_fit annotations
CONSDB_URL       = "auto"               # aos_trim.make_consdb_client endpoint selector
OFC_VERSION      = "v13"                # OFC config version for the sensitivity SVD

# Output
from pathlib import Path
OUTPUT_DIR = Path("output") / "danish_tarts_compare_20260713"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TAG = f"danish_tarts_{DAY_OBS}"


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# repo paths: aos/code for the analysis modules, repo root for common/
sys.path.insert(0, str(Path.cwd() / "code"))
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
setup_plotting()

# ---- reuse existing repo code ------------------------------------------------
import aos_state                         # canonical v-mode engine + DOF/Zernike sets
from aos_state import (ZK_NOLL, SENSOR_NAMES, DOF_SETS,
                       build_geom_svd, recover_optical_state)
import aos_trim                          # Trim (aggregatedDoF) + ConsDB client
from run_wfs_mktable import get_unpaired_zernikes, _zk_um
from run_wfs_dof_compare import robust_fit, nmad
from astropy.table import Table as AstropyTable

# LSST stack
from lsst.daf.butler import Butler
from lsst.obs.lsst import LsstCam
from lsst.ts.intrinsic.wavefront.intrinsics_lib import get_aggregate_zernikes
from lsst.ts.intrinsic.wavefront.ofc_svd import build_ofc_svd

CAMERA = LsstCam.getCamera()
NZ = len(ZK_NOLL)                        # 21 pupil Zernikes (Z4-Z26 excl Z20,Z21)
ZK_LABELS = [f"Z{j}" for j in ZK_NOLL]
print(f"ZK_NOLL ({NZ}): {ZK_NOLL}")
print(f"SENSOR_NAMES: {SENSOR_NAMES}")


<a id='functions'></a>
## Helper Functions

Thin glue over the reused modules. Plotters mirror `run_wfs_dof_compare.th_pages` / `scatter_pages` but display inline and overlay an arbitrary number of methods.

In [ ]:
def list_seqnums(butler, day_obs, dataset_type):
    """Available seq_nums for a dataset type on a night.

    seq_num / day_obs are exposure/visit RECORD attributes, not dimension keys,
    and queryDatasets dataIds come back without records attached (-> hasRecords()
    is False). Use queryDimensionRecords filtered to the dataset, which returns
    fully-populated records with .seq_num. The aggregateAOSVisitTable* products
    are visit-dimensioned, so try visit first, then exposure.
    """
    for dim in ("visit", "exposure"):
        try:
            recs = butler.registry.queryDimensionRecords(
                dim, datasets=dataset_type,
                where=f"instrument='LSSTCam' AND {dim}.day_obs={int(day_obs)}")
            seqs = sorted({int(r.seq_num) for r in recs})
            if seqs:
                return seqs
        except Exception as e:
            print(f"  [list_seqnums/{dim}] {dataset_type}: {e}")
    return []


def _align_to_noll(zk_arr, noll):
    """Reorder columns of (ndon, nz_file) zk to ZK_NOLL; drop missing -> NaN col."""
    idx = [noll.index(j) if j in noll else None for j in ZK_NOLL]
    out = np.full((zk_arr.shape[0], NZ), np.nan)
    for c, i in enumerate(idx):
        if i is not None:
            out[:, c] = zk_arr[:, i]
    return out


def load_visit(butler, day_obs, seq_num, reader, dataset_type, coord=COORD):
    """Return a tidy per-donut table for one visit/method.

    Reuses get_aggregate_zernikes (paired) / get_unpaired_zernikes (unpaired).
    Output dict: detector(str array), raft(str array), zk(ndon,NZ),
    zk_int(ndon,NZ), dev(ndon,NZ) — all microns, aligned to ZK_NOLL. None if absent.
    """
    zc, zic = f"zk_{coord}", f"zk_intrinsic_{coord}"
    if reader == "paired":
        tbl, meta = get_aggregate_zernikes(butler, day_obs, seq_num, coord, CAMERA)
    else:
        tbl, meta = get_unpaired_zernikes(butler, day_obs, seq_num, coord,
                                          dataset_type=dataset_type)
    if tbl is None or len(tbl) == 0 or meta is None:
        return None
    noll = [int(x) for x in meta["nollIndices"]]
    zk = _align_to_noll(np.stack([_zk_um(r) for r in tbl[zc]]), noll)   # microns
    if zic in tbl.colnames:
        zint = _align_to_noll(np.stack([_zk_um(r) for r in tbl[zic]]), noll)
    else:
        zint = np.zeros_like(zk)
    det = np.array([str(d) for d in tbl["detector"]]) if "detector" in tbl.colnames \
        else np.array(["UNK"] * len(tbl))
    raft = np.array([d.split("_")[0] for d in det])
    return dict(detector=det, raft=raft, zk=zk, zk_int=zint, dev=zk - zint)


def corner_medians(visit, key="zk", side=None):
    """(4, NZ) per-corner median of `key`, ordered by SENSOR_NAMES rafts.

    side: None (all donuts), or a chip suffix ('SW0'/'SW1') to keep one focus side.
    NaN row where a corner has no donut.
    """
    out = np.full((len(SENSOR_NAMES), NZ), np.nan)
    if visit is None:
        return out
    rafts = np.array([s.split("_")[0] for s in SENSOR_NAMES])
    m_side = (np.ones(len(visit["detector"]), bool) if side is None
              else np.char.endswith(visit["detector"], side))
    for i, raft in enumerate(rafts):
        m = (visit["raft"] == raft) & m_side
        if m.any():
            out[i] = np.nanmedian(visit[key][m], axis=0)
    return out


def corner_medians_sidemean(visit, key="zk"):
    """(4, NZ) straight mean of the intra-side and extra-side per-corner medians.

    This is the genuine mean of the two focus-side curves (equal weight to each
    side), as opposed to a median over the pooled intra+extra donuts.
    """
    a = corner_medians(visit, key=key, side=UNPAIRED_INTRA_SUFFIX)
    b = corner_medians(visit, key=key, side=UNPAIRED_EXTRA_SUFFIX)
    return np.nanmean(np.stack([a, b]), axis=0)


def corner_cube(name, key, view):
    """(n_seq, 4, NZ) per-corner medians for method `name`, column `key`
    ('zk' | 'zk_int' | 'dev'), across `common_seq`.

    view: 'both'/'pooled' (all donuts in the raft), 'intra'/'extra' (one focus
    side), 'sidemean' (straight mean of the two focus sides).
    """
    def cm(v):
        if view in ("both", "pooled"):
            return corner_medians(v, key=key, side=None)
        if view == "intra":
            return corner_medians(v, key=key, side=UNPAIRED_INTRA_SUFFIX)
        if view == "extra":
            return corner_medians(v, key=key, side=UNPAIRED_EXTRA_SUFFIX)
        if view == "sidemean":
            return corner_medians_sidemean(v, key=key)
        raise ValueError(f"unknown view {view!r}")
    return np.stack([cm(raw[name][s]) for s in common_seq])


# ---- v-mode engine (reuses aos_state) ---------------------------------------
def build_vmode_svds(version=OFC_VERSION):
    """One geom-normalized sensitivity SVD per scheme (reuses aos_state.build_geom_svd)."""
    return {name: build_geom_svd(dof_set=dof_set, version=version)
            for name, (dof_set, _n) in VMODE_SCHEMES.items()}


def vmodes_from_corner_dev(dev_corner, svd, n_modes):
    """v-mode amplitudes from a (4, NZ) per-corner DEVIATION wavefront.

    Flatten corner-major (SENSOR_NAMES order) to the 4*NZ SVD row order and run
    aos_state.recover_optical_state (least-squares onto the top-n_modes subspace).
    Returns (n_modes,) or NaN if any corner is missing.
    """
    if not np.all(np.isfinite(dev_corner)):
        return np.full(n_modes, np.nan)
    _dof, vamp, _zc = recover_optical_state(dev_corner.ravel(), svd, n_modes=n_modes)
    return vamp


# ---- inline plotters (mirror run_wfs_dof_compare.th_pages / scatter_pages) ---
def timehist_pages(labels, x, series, title, pdf=None, per=8, xlabel="seq_num"):
    """One panel per term, `per` panels/page; overlay each method in `series`.

    series: list of (label, (n_x, n_terms) array, color). Displays inline; also
    writes to `pdf` (PdfPages) if given.
    """
    n = len(labels)
    for p0 in range(0, n, per):
        idx = list(range(p0, min(p0 + per, n)))
        nr = int(np.ceil(len(idx) / 2)); nc = 2
        fig, axes = plt.subplots(nr, nc, figsize=(13, 2.4 * nr),
                                 constrained_layout=True, squeeze=False)
        for ax, i in zip(axes.ravel(), idx):
            for lab, arr, c in series:
                ax.plot(x, arr[:, i], "-o", ms=2.5, lw=0.7, color=c, label=lab)
            ax.axhline(0, color="k", lw=0.3)
            ax.set_title(labels[i], fontsize=8); ax.tick_params(labelsize=7)
            ax.set_xlabel(xlabel, fontsize=7)
        for ax in axes.ravel()[len(idx):]:
            ax.axis("off")
        axes.ravel()[0].legend(fontsize=6, ncol=1)
        fig.suptitle(f"{title}  [page {p0//per + 1}]", fontsize=11)
        if pdf is not None:
            pdf.savefig(fig)
        plt.show()


def scatter_pages(labels, x, y, xlabel, ylabel, title, pdf=None, per=None, K=SCATTER_REJECT_K):
    """method-x vs method-y scatter, one panel per term (all on one page by default).

    x, y: (n_points, n_terms). Robust y=x + fit line + annotation via
    run_wfs_dof_compare.robust_fit. Displays inline; writes to `pdf` if given.
    """
    n = len(labels)
    per = per or n
    for p0 in range(0, n, per):
        idx = list(range(p0, min(p0 + per, n)))
        nc = 3 if len(idx) > 4 else 2
        nr = int(np.ceil(len(idx) / nc))
        fig, axes = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.6 * nr),
                                 constrained_layout=True, squeeze=False)
        for ax, i in zip(axes.ravel(), idx):
            xi, yi = x[:, i], y[:, i]
            keep, m = robust_fit(xi, yi, K)
            fm = np.isfinite(xi) & np.isfinite(yi)
            xf, yf = xi[fm], yi[fm]
            ax.scatter(xf[keep], yf[keep], s=8, alpha=0.5)
            if (~keep).any():
                ax.scatter(xf[~keep], yf[~keep], s=8, alpha=0.4, color="lightgray")
            if np.isfinite(m["slope"]) and keep.any():
                lo, hi = np.nanpercentile(np.concatenate([xf[keep], yf[keep]]), [1, 99])
                ax.plot([lo, hi], [lo, hi], "k--", lw=0.6)
                ax.plot([lo, hi], [m["slope"] * lo + m["off"], m["slope"] * hi + m["off"]],
                        "r-", lw=0.9)
                ax.text(0.04, 0.96,
                        f"r={m['r']:.2f} s={m['slope']:.2f}\noff={m['off']:.3f} "
                        f"rms={m['rms']:.3f}\nn={m['n']} drop={m['ndrop']}",
                        transform=ax.transAxes, va="top", fontsize=6)
            ax.set_title(labels[i], fontsize=8); ax.tick_params(labelsize=7)
            ax.set_xlabel(xlabel, fontsize=7); ax.set_ylabel(ylabel, fontsize=7)
        for ax in axes.ravel()[len(idx):]:
            ax.axis("off")
        fig.suptitle(title, fontsize=12)
        if pdf is not None:
            pdf.savefig(fig)
        plt.show()


<a id='data'></a>
## Data Access — load per-method corner Zernikes

Open a Butler per method, find the common `seq_num`s on the night, and load every visit. Per visit we keep both the per-corner medians (for v-modes) and the corner-averaged mean Zernike (for time-histories / scatter).

In [ ]:
# --- open butlers + discover seq_nums per method ---
butlers, seq_by_method = {}, {}
for name, cfg in METHODS.items():
    butlers[name] = Butler(BUTLER_REPO, collections=cfg["collection"])
    seq_by_method[name] = list_seqnums(butlers[name], DAY_OBS, cfg["dataset_type"])
    print(f"{name:16s}: {len(seq_by_method[name])} seq_nums")

# common seq_nums across all three methods (aligned comparison)
common_seq = sorted(reduce(lambda a, b: a & b,
                           (set(s) for s in seq_by_method.values())))
print(f"\nCommon seq_nums: {len(common_seq)}  "
      f"[{common_seq[:3]} ... {common_seq[-3:]}]" if common_seq else "NONE")
SEQ = np.array(common_seq)


In [ ]:
# --- load every visit for every method ---
# raw[name][seq] = load_visit dict (or None). Also precompute the arrays we plot.
raw = {name: {} for name in METHODS}
for name, cfg in METHODS.items():
    for s in common_seq:
        raw[name][s] = load_visit(butlers[name], DAY_OBS, int(s),
                                  cfg["reader"], cfg["dataset_type"])
    n_ok = sum(v is not None for v in raw[name].values())
    print(f"{name:16s}: loaded {n_ok}/{len(common_seq)} visits")


In [ ]:
# --- per-corner cubes (n_seq, 4, NZ) for every plotted series, microns ---
# Danish unpaired "(mean)" uses the straight mean of the two focus sides
# (UNPAIRED_MEAN_MODE); the intrinsic series is the TARTS tabulated intrinsic.
UNP_MEAN_VIEW = "sidemean" if UNPAIRED_MEAN_MODE == "sidemean" else "pooled"
SERIES_CUBE = {
    "TARTS":                    corner_cube("TARTS", "zk", "both"),
    "Danish paired":            corner_cube("Danish paired", "zk", "both"),
    "Danish unpaired (intra)":  corner_cube("Danish unpaired", "zk", "intra"),
    "Danish unpaired (extra)":  corner_cube("Danish unpaired", "zk", "extra"),
    "Danish unpaired (mean)":   corner_cube("Danish unpaired", "zk", UNP_MEAN_VIEW),
    "Intrinsic (TARTS)":        corner_cube("TARTS", "zk_int", "both"),
}
# corner-averaged series (mean over the 4 corners) for the standard time-history
zk_series = {k: np.nanmean(v, axis=1) for k, v in SERIES_CUBE.items()}
zk_colors = {
    "TARTS": "crimson", "Danish paired": "steelblue",
    "Danish unpaired (intra)": "darkorange", "Danish unpaired (extra)": "seagreen",
    "Danish unpaired (mean)": "black", "Intrinsic (TARTS)": "dimgray",
}
CORNER_RAFTS = [s.split("_")[0] for s in SENSOR_NAMES]     # R00 R04 R40 R44
for k, v in zk_series.items():
    print(f"{k:26s}: {np.isfinite(v).all(axis=1).sum()}/{len(v)} finite visits")


<a id='zk-timehist'></a>
## 1. Zernike time histories vs `seq_num`

One panel per Zernike (Z4–Z26 excl. Z20,Z21). All methods overlaid; Danish unpaired shown as intra / extra / mean.

In [ ]:
pdf_path = OUTPUT_DIR / f"{TAG}_zk_timehist.pdf"
series = [(lab, zk_series[lab], zk_colors[lab]) for lab in zk_series]
with PdfPages(pdf_path) as pdf:
    timehist_pages(ZK_LABELS, SEQ, series,
                   f"Zernike time history ({COORD}, microns) — {DAY_OBS}",
                   pdf=pdf, per=8)
print("wrote", pdf_path)


### 1b. Per-corner Zernike time histories

Same overlays, but for each corner raft individually (R00, R04, R40, R44) rather than the corner-average — one multi-page PDF per corner.

In [ ]:
if PLOT_PER_CORNER:
    for ci, raft in enumerate(CORNER_RAFTS):
        series_c = [(lab, SERIES_CUBE[lab][:, ci, :], zk_colors[lab])
                    for lab in SERIES_CUBE]
        pdf_path = OUTPUT_DIR / f"{TAG}_zk_timehist_{raft}.pdf"
        with PdfPages(pdf_path) as pdf:
            timehist_pages(ZK_LABELS, SEQ, series_c,
                           f"Zernike time history — corner {raft} "
                           f"({COORD}, microns) — {DAY_OBS}", pdf=pdf, per=8)
        print("wrote", pdf_path)


<a id='vmode-timehist'></a>
## 2. v-modes — compute + time histories

v-modes are computed with the canonical `aos_state` engine (`build_geom_svd` + `recover_optical_state`) from each method's **per-corner deviation** wavefront (OPD − intrinsic), for both the (22 DOF, 12 v-mode) and (50 DOF, 34 v-mode) schemes. For each method we use the corner-median deviation; Danish unpaired uses the mean over the two focus sides.

In [ ]:
SVDS = build_vmode_svds()
for name, svd in SVDS.items():
    print(f"{name}: {len(svd['dof_indices'])} DOF, {svd['n_modes']} singular values")

# per-method (n_seq, 4, NZ) corner DEVIATION = measured - intrinsic.
# The intrinsic is the same for all methods when INTRINSIC_SOURCE == "TARTS"
# (the intrinsic tabulated in the TARTS aggregate table); otherwise each method
# subtracts its own tabulated intrinsic. Danish unpaired uses the straight
# side-mean of its measured Zernikes (matching the "(mean)" series).
VMODE_MEAS_VIEW = {"TARTS": "both", "Danish paired": "both",
                   "Danish unpaired": UNP_MEAN_VIEW}
_intr_tarts = corner_cube("TARTS", "zk_int", "both")        # common intrinsic

def dev_cube(name):
    meas = corner_cube(name, "zk", VMODE_MEAS_VIEW[name])
    intr = _intr_tarts if INTRINSIC_SOURCE == "TARTS" \
        else corner_cube(name, "zk_int", VMODE_MEAS_VIEW[name])
    return meas - intr

DEV = {name: dev_cube(name) for name in METHODS}
print(f"v-mode deviations use intrinsic source: {INTRINSIC_SOURCE}")

# vmodes[scheme][method] = (n_seq, n_modes)
VM = {sch: {} for sch in VMODE_SCHEMES}
for sch, (dof_set, n_modes) in VMODE_SCHEMES.items():
    svd = SVDS[sch]
    for name in METHODS:
        VM[sch][name] = np.vstack([
            vmodes_from_corner_dev(DEV[name][i], svd, n_modes)
            for i in range(len(common_seq))])
    print(f"{sch}: computed v-modes for {list(METHODS)}")


In [ ]:
# v-mode time histories: 12 panels (1 page) for 22/12; 34 panels (multi-page) for 50/34
vm_colors = {name: METHODS[name]["color"] for name in METHODS}
for sch, (dof_set, n_modes) in VMODE_SCHEMES.items():
    labels = [f"v{m+1}" for m in range(n_modes)]
    series = [(name, VM[sch][name], vm_colors[name]) for name in METHODS]
    pdf_path = OUTPUT_DIR / f"{TAG}_vmode_timehist_{sch}.pdf"
    with PdfPages(pdf_path) as pdf:
        timehist_pages(labels, SEQ, series,
                       f"v-mode time history — {sch} ({DAY_OBS})",
                       pdf=pdf, per=12)
    print("wrote", pdf_path)


<a id='zk-scatter'></a>
## 3. Zernike scatter: TARTS vs Danish paired

One panel per Zj (21 panels, one page). Robust `y=x` and fit annotation via `run_wfs_dof_compare.robust_fit`.

In [ ]:
x = zk_series["Danish paired"]     # x-axis
y = zk_series["TARTS"]             # y-axis
pdf_path = OUTPUT_DIR / f"{TAG}_zk_scatter_tarts_vs_danishpaired.pdf"
with PdfPages(pdf_path) as pdf:
    scatter_pages(ZK_LABELS, x, y,
                  xlabel="Danish paired [um]", ylabel="TARTS [um]",
                  title=f"TARTS vs Danish paired — Zernikes ({COORD}, {DAY_OBS})",
                  pdf=pdf)
print("wrote", pdf_path)


<a id='vmode-scatter'></a>
## 4. v-mode scatter: TARTS vs Danish paired

Both schemes (12 and 34 v-modes).

In [ ]:
for sch, (dof_set, n_modes) in VMODE_SCHEMES.items():
    labels = [f"v{m+1}" for m in range(n_modes)]
    x = VM[sch]["Danish paired"]
    y = VM[sch]["TARTS"]
    pdf_path = OUTPUT_DIR / f"{TAG}_vmode_scatter_tarts_vs_danishpaired_{sch}.pdf"
    with PdfPages(pdf_path) as pdf:
        scatter_pages(labels, x, y,
                      xlabel="Danish paired", ylabel="TARTS",
                      title=f"TARTS vs Danish paired — v-modes {sch} ({DAY_OBS})",
                      pdf=pdf)
    print("wrote", pdf_path)


<a id='trim-fwhm'></a>
## 5. Trim (22 DOF) & delivered FWHM vs `seq_num`

**Trim** = the closed-loop `aggregatedDoF` (EFD, anchored on ConsDB exposure time) via `aos_trim.fetch_aggregated_dof_for_visits`, sliced to the 22-DOF set. **Delivered FWHM** from ConsDB `visit1_quicklook.psf_sigma_median` (× 2.3548 × 0.2 arcsec).

In [ ]:
# --- Trim of the 22 DOF vs seq_num ---
# NB: fetch_aggregated_dof_for_visits uses .colnames -> pass an astropy Table.
fit_table = AstropyTable({"day_obs": np.full(len(common_seq), DAY_OBS, int),
                          "seq_num": np.array(common_seq, int)})
trim50, trim_info = aos_trim.fetch_aggregated_dof_for_visits(fit_table)
dof22_idx = DOF_SETS["standard_22"]
trim22 = trim50[:, dof22_idx]                       # (n_seq, 22)

# human-readable DOF labels (reuse ofc_svd dof_labels for the 22-DOF subset)
_svd_lab = build_ofc_svd(list(ZK_NOLL), k_min=1, k_max=6, n_keep=12,
                         n_dof=list(dof22_idx))
dof22_labels, dof22_units = _svd_lab.dof_labels()

pdf_path = OUTPUT_DIR / f"{TAG}_trim22_timehist.pdf"
with PdfPages(pdf_path) as pdf:
    timehist_pages([f"{l} [{u}]" for l, u in zip(dof22_labels, dof22_units)],
                   SEQ, [("Trim (aggregatedDoF)", trim22, "purple")],
                   f"22-DOF Trim time history ({DAY_OBS})", pdf=pdf, per=8)
print("wrote", pdf_path)
ev = np.asarray(trim_info.get("event_id", []))          # mark AOS re-alignments
n_realign = int(np.sum(ev[1:] != ev[:-1])) if ev.size > 1 else 0
print(f"Trim re-alignments (event_id changes across the {len(SEQ)} visits): {n_realign}")


In [ ]:
# --- delivered FWHM from ConsDB vs seq_num ---
cdb = aos_trim.make_consdb_client(CONSDB_URL)
seq_list = ",".join(str(int(s)) for s in common_seq)
q = f"""
    SELECT v.visit_id, v.seq_num, v.day_obs, v.band,
           ql.psf_sigma_median, ql.seeing_zenith_500nm_median,
           ql.aos_fwhm, ql.donut_blur_fwhm
    FROM cdb_lsstcam.visit1 AS v
    LEFT JOIN cdb_lsstcam.visit1_quicklook AS ql ON ql.visit_id = v.visit_id
    WHERE v.day_obs = {int(DAY_OBS)} AND v.seq_num IN ({seq_list})
    ORDER BY v.seq_num
"""
fwhm = cdb.query(q).to_pandas()
for c in ["psf_sigma_median", "seeing_zenith_500nm_median", "aos_fwhm", "donut_blur_fwhm"]:
    fwhm[c] = pd.to_numeric(fwhm[c], errors="coerce")
fwhm["psf_fwhm_median"] = 2.3548 * fwhm["psf_sigma_median"] * 0.2   # arcsec

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
ax.plot(fwhm["seq_num"], fwhm["psf_fwhm_median"], "-o", ms=3, color="navy",
        label="delivered FWHM (psf_sigma_median)")
if fwhm["aos_fwhm"].notna().any():
    ax.plot(fwhm["seq_num"], fwhm["aos_fwhm"], "-s", ms=3, color="darkorange",
            label="aos_fwhm")
if fwhm["donut_blur_fwhm"].notna().any():
    ax.plot(fwhm["seq_num"], fwhm["donut_blur_fwhm"], "-^", ms=3, color="seagreen",
            label="donut_blur_fwhm")
ax.set_xlabel("seq_num"); ax.set_ylabel("FWHM [arcsec]")
ax.set_title(f"Delivered FWHM (ConsDB) — {DAY_OBS}"); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(OUTPUT_DIR / f"{TAG}_delivered_fwhm.pdf")
plt.show()
print(f"median delivered FWHM = {fwhm['psf_fwhm_median'].median():.3f} arcsec")
